# Compare methods vs exact
Reads `results/<circuit>/*.json` written by `run_circuit.py` / `exact_local.py`.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from common import RESULTS
from score import corr2

CIRCUIT = "S1"
REF = "exact_local"

runs = {p.stem: json.loads(p.read_text())["values"] for p in sorted((RESULTS / CIRCUIT).glob("*.json"))}
qid = lambda k: int(k.rsplit("_", 1)[1])
list(runs)

In [ ]:
# value per qubit for every run
fig, ax = plt.subplots(figsize=(10, 4))
for tag, vals in runs.items():
    ks = sorted(vals, key=qid)
    ax.plot([qid(k) for k in ks], [vals[k] for k in ks], "o-" if tag == REF else ".--", label=tag)
ax.set(xlabel="qubit q", ylabel="<Z_q>", title=CIRCUIT)
ax.legend(fontsize=8)
plt.show()

In [ ]:
# scatter each approximate run against the exact values
others = [t for t in runs if t != REF]
if REF in runs and others:
    fig, axes = plt.subplots(1, len(others), figsize=(4 * len(others), 4), squeeze=False)
    for ax, tag in zip(axes[0], others):
        ks = sorted(set(runs[tag]) & set(runs[REF]), key=qid)
        x = [runs[REF][k] for k in ks]; y = [runs[tag][k] for k in ks]
        ax.scatter(x, y, s=15)
        ax.set(title=f"{tag}\ncorr²={corr2(x, y):.3f}", xlabel="exact", ylabel="estimate")
    plt.tight_layout(); plt.show()